In [1]:
import os, sys
os.environ['HF_ENDPOINT'] = 'https://alpha.hf-mirror.com/'
os.environ["HF_HUB_URL"] = 'https://alpha.hf-mirror.com/'


In [2]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from janus.models import MultiModalityCausalLM, VLChatProcessor
from janus.utils.io import load_pil_images

if torch.cuda.is_available():
    torch.device('cuda')
else:
    torch.device('cpu')

torch.cuda.empty_cache()

bnb_4bit_config = BitsAndBytesConfig(
    load_in_4bit=True,  
    bnb_4bit_compute_dtype=torch.float16,   
    bnb_4bit_use_double_quant=True,         
    bnb_4bit_quant_type="nf4",              # or "fp4"
)

# specify the path to the model
model_path = "deepseek-ai/Janus-Pro-1B"
vl_chat_processor: VLChatProcessor = VLChatProcessor.from_pretrained(model_path, 
                                                                     quantization_config=bnb_4bit_config,
                                                                     torch_dtype=torch.bfloat16,
                                                                     attn_implementation="flash_attention_2")
tokenizer = vl_chat_processor.tokenizer

vl_gpt: MultiModalityCausalLM = AutoModelForCausalLM.from_pretrained(
    model_path, trust_remote_code=True
)
vl_gpt = vl_gpt.to(torch.bfloat16).cuda().eval()



/root/miniconda3/envs/janus/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python version is above 3.10, patching the collections module.


/root/miniconda3/envs/janus/lib/python3.13/site-packages/transformers/models/auto/image_processing_auto.py:594: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughl

In [3]:
demo_conversation = [
    {
        "role": "User",
        "content": "<image_placeholder>\nConvert the table to HTML",
        "images": ["MMTab/pre_train_ds/images/table_pretrain_part_1/TABMWP_1.jpg"],
    },
    {"role": "Assistant", "content": ""},
]

# load images and prepare for inputs
pil_images = load_pil_images(demo_conversation)
prepare_inputs = vl_chat_processor(
    conversations=demo_conversation, images=pil_images, force_batchify=True
).to(vl_gpt.device)
# # run image encoder to get the image embeddings
inputs_embeds = vl_gpt.prepare_inputs_embeds(**prepare_inputs)
print(inputs_embeds.shape)

# # run the model to get the response
outputs = vl_gpt.language_model.generate(
    inputs_embeds=inputs_embeds,
    attention_mask=prepare_inputs.attention_mask,
    pad_token_id=tokenizer.eos_token_id,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    max_new_tokens=2048,
    do_sample=False,
    use_cache=True,
)

answer = tokenizer.decode(outputs[0].cpu().tolist(), skip_special_tokens=True)
print(f"{prepare_inputs['sft_format'][0]}", answer)

torch.Size([1, 629, 2048])
You are a helpful language and vision assistant. You are able to understand the visual content that the user provides, and assist the user with a variety of tasks using natural language.

User: <image_placeholder>
Convert the table to HTML

Assistant: Sure, here is the HTML representation of the table:

```html
<table>
  <tr>
    <th>Stem</th>
    <th>Leaf</th>
  </tr>
  <tr>
    <td>3</td>
    <td>33355</td>
  </tr>
  <tr>
    <td>4</td>
    <td>6</td>
  </tr>
  <tr>
    <td>4</td>
    <td>6</td>
  </tr>
  <tr>
    <td>5</td>
    <td>4578</td>
  </tr>
  <tr>
    <td>6</td>
    <td>78</td>
  </tr>
  <tr>
    <td>6</td>
    <td>78</td>
  </tr>
  <tr>
    <td>7</td>
    <td>2379</td>
  </tr>
  <tr>
    <td>8</td>
    <td>689</td>
  </tr>
</table>
```


In [4]:
print(vl_gpt)

MultiModalityCausalLM(
  (vision_model): CLIPVisionTower(
    (vision_tower): VisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
        (norm): Identity()
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (patch_drop): Identity()
      (norm_pre): Identity()
      (blocks): Sequential(
        (0): Block(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Attention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
            (proj_drop): Identity()
          )
          (ls1): Identity()
          (drop_path1): Identity()
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in

In [5]:
from datasets import load_dataset

pre_train_ds_path = "MMTab/pre_train_ds/MMTab-pre_pretrain_data_llava_format_150K.json"
dataset = load_dataset("json", data_files=pre_train_ds_path, streaming=False)["train"]
print(dataset)

Dataset({
    features: ['id', 'image', 'conversations'],
    num_rows: 150521
})


In [6]:
def process_fn(row):
	images = row['image']
	conversations = row['conversations']
	row['janus_conversation'], row['labels'] = build_janus_template(images, conversations)
	return row


def build_janus_template(image, conversation):
	# convert one row of llava format to janus format
	llava_image_prompts = conversation[0]['value'].split('\n')
	prompt = llava_image_prompts[0] if llava_image_prompts[0].strip() != '<image>' else llava_image_prompts[1]
	return [
		{
			"role": "User",
			"content": f"<image_placeholder>\n{prompt}",
			"images": [f"MMTab/pre_train_ds/images/{image}"],
		},
		{"role": "Assistant", "content": ""},
	], conversation[1]['value']

test_dataset = dataset.select([1, 2, 3, 4, 5]).map(process_fn, batched=False, remove_columns=dataset.column_names)
print(test_dataset)
print(test_dataset[0])
# processed_dataset = dataset.map(process_fn, batched=False, remove_columns=dataset.column_names)

Dataset({
    features: ['janus_conversation', 'labels'],
    num_rows: 5
})
{'janus_conversation': [{'content': '<image_placeholder>\nPlease read the table in this image and return an HTML-style reconstructed table in text.', 'images': ['MMTab/pre_train_ds/images/table_pretrain_part_1/TABMWP_4.jpg'], 'role': 'User'}, {'content': '', 'images': None, 'role': 'Assistant'}], 'labels': '<table border="1" cellspacing="0">\n<tr> <th colspan="2"> Train tickets sold </th> </tr><tr> <th> Day </th> <th> Number of tickets </th> </tr>\n<tr> <td> Friday </td> <td> 71 </td> </tr>\n<tr> <td> Saturday </td> <td> 74 </td> </tr>\n<tr> <td> Sunday </td> <td> 75 </td> </tr>\n<tr> <td> Monday </td> <td> 72 </td> </tr>\n</table>'}


In [7]:

def collate_fn(rows):
	batch_size = len(rows)
	if batch_size > 0:
		vl_chat_processor_outputs = []
		labels = []
		for row in rows:
			# load images and prepare for inputs
			pil_images = load_pil_images(row["janus_conversation"])
			vl_chat_processor_output = vl_chat_processor(conversations=row["janus_conversation"], images=pil_images, force_batchify=False)
			vl_chat_processor_outputs.append(vl_chat_processor_output)
			labels.append(row["labels"])
		
		
		tokenized_labels = tokenizer(labels, padding=True, truncation=False, return_tensors="pt", padding_side="right").to(vl_gpt.device)
		answer_ids = tokenized_labels.input_ids.to(vl_gpt.device)
		batched_prepare = vl_chat_processor.batchify(vl_chat_processor_outputs).to(vl_gpt.device)
		context_ids = 
		context_len = batched_prepare.input_ids.shape[-1]
		answer_len = answer_ids.shape[-1]
		seq_len = context_len + answer_len
		padded_labels = torch.zeros([batch_size, seq_len])
		padded_labels[:, :answer_len] = -100
		padded_labels[:, answer_len:] = answer_ids
		padded_input = torch.zeros([batch_size, seq_len])
		padded_input[:, context_len:] = 

		batch = {
			"input_ids":batched_prepare.input_ids,
			"attention_mask":batched_prepare.attention_mask,
			"pixel_values":batched_prepare.pixel_values,
			"images_seq_mask":batched_prepare.images_seq_mask,
			"images_emb_mask":batched_prepare.images_emb_mask,
			"sft_format":batched_prepare.sft_format,
			"labels": labels_tensor,
		}
	return batch

In [8]:
# import torch.nn as nn

# class VLMWrapper(nn.Module):
#     def __init__(self, vl_gpt):
#         super().__init__()
#         self.vl_gpt = vl_gpt

#     def forward(self, batch, input_ids=None, labels=None, **kwargs):
#         batched_prepare = batch["batched_prepare"]
#         inputs_embeds = vl_gpt.prepare_inputs_embeds(**batched_prepare)
#         labels=batch["labels"]

#         llm_outputs = self.vl_gpt.language_model(             
#             inputs_embeds=inputs_embeds,       
#             attention_mask=batched_prepare.attention_mask,      
#             labels=labels,                     
#             **kwargs
#         )
        
#         return llm_outputs  # includes .loss, .logits, etc.

# vlm_wrapper = VLMWrapper(vl_gpt)

In [10]:
from transformers import Trainer, TrainingArguments
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training


lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],  # typical for LLaMA
)
# lora freeze all param 
vl_gpt = prepare_model_for_kbit_training(vl_gpt)

vl_gpt = get_peft_model(vl_gpt, lora_config)


for param in vl_gpt.aligner.parameters():
    param.requires_grad = True

training_args = TrainingArguments(
    output_dir="./ckpt-janus1b-4bit-output",
    max_steps=10000,
    per_device_train_batch_size=2,
    evaluation_strategy="no",
    # per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    # evaluation_strategy="epoch",
    save_strategy="steps",             
    save_steps=500,
    learning_rate=2e-4,
    bf16=True,  
    logging_steps=10,
    remove_unused_columns=False,
    dataloader_pin_memory=False
)

trainer = Trainer(
    model=vl_gpt,
    args=training_args,
    train_dataset=test_dataset,
    data_collator=collate_fn,
)

trainer.train()

/root/miniconda3/envs/janus/lib/python3.13/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


torch.Size([2, 209])
torch.Size([2, 642, 2048])


ValueError: Expected input batch_size (1284) to match target batch_size (418).

In [ ]:
trainable_params = 0
all_params = 0
for name, param in vl_gpt.named_parameters():
    all_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()
        print(f"Trainable: {name} => shape={param.shape}")
    else:
        print(f"Frozen: {name}")

print(f"Trainable = {trainable_params} / {all_params} params => {100 * trainable_params/all_params:.2f}%")


In [11]:
vl_gpt.print_trainable_parameters()

trainable params: 7,868,416 || all params: 2,090,804,875 || trainable%: 0.3763
